## Scraping AWOIAF Characters
End-to-end scrape of character data from [A Wiki of Ice and Fire](https://awoiaf.westeros.org):

1. **List**: pull every character name from `List_of_characters` → `characters.csv`
2. **Enrich**: visit each character's page, parse the infobox (`father`, `mother`, `spouse`, `lover`, `issue`, `allegiance`) and collect every internal `/index.php/` link from the article body into `affiliated` (used as edges for the network analysis) → `characters_enriched.csv`

In [18]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from urllib.parse import quote, unquote
from concurrent.futures import ThreadPoolExecutor

### Setup
Full browser User-Agent — the wiki returns 403 for generic bot strings.

The label map accepts both singular and plural variants (`Allegiance`/`Allegiances`, `Lover`/`Lovers`/`Paramour`) because the wiki uses both inconsistently — mapping only the singular dropped data for ~1600 character pages.

In [ ]:
BASE = 'https://awoiaf.westeros.org'
PREFIX = '/index.php/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

# Infobox label -> normalized column name. The wiki uses both 'Allegiance' and 'Allegiances',
# and both 'Lover'/'Lovers'/'Paramour'; map every variant so no rows are silently dropped.
INFOBOX_LABELS = {
    'Father': 'father',
    'Fathers': 'father',
    'Mother': 'mother',
    'Mothers': 'mother',
    'Spouse': 'spouse',
    'Spouses': 'spouse',
    'Lover': 'lover',
    'Lovers': 'lover',
    'Paramour': 'lover',
    'Issue': 'issue',
    'Issues': 'issue',
    'Allegiance': 'allegiance',
    'Allegiances': 'allegiance',
}

session = requests.Session()
session.headers.update(headers)

## 1. Scrape the character list
Character entries on `List_of_characters` appear as `<li>` items inside `mw-parser-output`, each wrapped in an `<a>` to a `/index.php/...` page. Many characters share a display name (e.g. several "Aegon Targaryen"s) but each has a unique page slug like `Aegon_Targaryen_(son_of_Gaemon)`.

We therefore **dedupe on page slug (`ID`), not on display name** — deduping on display name silently merges all the Aegons into one row. The URL-decoded slug is kept as the canonical `ID`; it doubles as the lookup key for the network analysis.

In [ ]:
list_page = session.get(BASE + PREFIX + 'List_of_characters', timeout=20)
print(f'Status code: {list_page.status_code}')

soup = BeautifulSoup(list_page.content, 'html.parser')
content = soup.find('div', class_='mw-parser-output')

characters = []  # list of {'name', 'ID'} keyed by ID so disambiguated pages aren't merged
seen_ids = set()
for li in content.find_all('li'):
    a = li.find('a')
    if not a:
        continue
    href = a.get('href', '')
    if not href.startswith(PREFIX):
        continue
    if 'redlink=1' in href or 'action=' in href:
        continue
    slug = href[len(PREFIX):].split('#', 1)[0]  # drop any #anchor
    if not slug or slug.startswith('Special:'):
        continue
    slug = unquote(slug)  # canonical form: apostrophes, not %27
    if slug in seen_ids:
        continue
    seen_ids.add(slug)
    name = a.get_text(strip=True)
    if not name:
        continue
    characters.append({'name': name, 'ID': slug})

pd.DataFrame(characters, columns=['name', 'ID']).to_csv('../csvs/characters.csv', index=False)
print(f'Saved {len(characters)} characters to characters.csv')
print(characters[:10])

Status code: 200
Saved 3689 characters to characters.csv
[{'name': 'A certain man', 'ID': 'A_certain_man'}, {'name': 'Abelar Hightower', 'ID': 'Abelar_Hightower'}, {'name': 'Abelon', 'ID': 'Abelon'}, {'name': 'Addam of Duskendale', 'ID': 'Addam_of_Duskendale'}, {'name': 'Addam Frey', 'ID': 'Addam_Frey'}, {'name': 'Addam Hightower', 'ID': 'Addam_Hightower'}, {'name': 'Addam Marbrand', 'ID': 'Addam_Marbrand'}, {'name': 'Addam Osgrey', 'ID': 'Addam_Osgrey'}, {'name': 'Addam Rivers', 'ID': 'Addam_Rivers'}, {'name': 'Addam Velaryon', 'ID': 'Addam_Velaryon'}]


## 2. Enrich each character
### Helpers

In [21]:
def slug_to_url(slug):
    """Turn a canonical (URL-decoded) slug into the full wiki URL.

    Re-encodes unsafe characters while keeping the structural punctuation the
    wiki itself uses in URLs (underscores, parens, commas, slashes, apostrophes)
    readable in the request line. MediaWiki accepts either `'` or `%27`.
    """
    return BASE + PREFIX + quote(slug, safe="_/(),'")


def href_to_slug(href):
    """Return the canonical (URL-decoded) slug for an internal wiki link, or
    None if the href isn't one we want. Drops URL fragments (`#anchor`), red
    links, edit/action links, and `Special:` pages.
    """
    if not href.startswith(PREFIX):
        return None
    if 'redlink=1' in href or 'action=' in href:
        return None
    slug = href[len(PREFIX):].split('#', 1)[0]
    if not slug or slug.startswith('Special:'):
        return None
    return unquote(slug)


def cell_links_or_text(td):
    """Semicolon-joined slugs in the cell. Falls back to plain text (e.g. 'Unknown') if none."""
    slugs = []
    seen = set()
    for a in td.find_all('a'):
        slug = href_to_slug(a.get('href', ''))
        if slug is None or slug in seen:
            continue
        seen.add(slug)
        slugs.append(slug)
    if slugs:
        return ';'.join(slugs)
    text = td.get_text(' ', strip=True)
    return text if text else ''


def parse_infobox(soup):
    data = {col: '' for col in INFOBOX_LABELS.values()}
    infobox = soup.find('table', class_='infobox')
    if infobox is None:
        return data
    for row in infobox.find_all('tr'):
        th = row.find('th')
        td = row.find('td')
        if not th or not td:
            continue
        label = th.get_text(strip=True).rstrip(':')
        col = INFOBOX_LABELS.get(label)
        if col is None:
            continue
        # first non-empty occurrence wins (handles e.g. both `Lover` and `Lovers` on one page)
        if data[col] == '':
            data[col] = cell_links_or_text(td)
    return data


def parse_affiliated(soup, self_slug):
    """All unique internal-link slugs in the article body, in document order, excluding the page itself."""
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return ''
    slugs = []
    seen = set()
    for a in content.find_all('a'):
        slug = href_to_slug(a.get('href', ''))
        if slug is None or slug == self_slug or slug in seen:
            continue
        seen.add(slug)
        slugs.append(slug)
    return ';'.join(slugs)

### Per-character scrape

In [22]:
EMPTY = {'father': '', 'mother': '', 'spouse': '', 'lover': '', 'issue': '',
         'allegiance': '', 'affiliated': ''}


def scrape_character(row):
    """Fetch and parse one character page. `row` is a dict with 'name' and 'ID' (canonical slug).

    We fetch by slug rather than by name because many display names are ambiguous
    (e.g. multiple 'Aegon Targaryen'); the slug is the unique page identifier.
    """
    name, slug = row['name'], row['ID']
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException as e:
        print(f'  ! request failed for {slug}: {e}')
        return {'name': name, 'ID': slug, **EMPTY}
    if resp.status_code != 200:
        print(f'  ! {resp.status_code} for {slug}')
        return {'name': name, 'ID': slug, **EMPTY}
    soup = BeautifulSoup(resp.content, 'html.parser')
    info = parse_infobox(soup)
    return {
        'name': name,
        'ID': slug,
        'father': info['father'],
        'mother': info['mother'],
        'spouse': info['spouse'],
        'lover': info['lover'],
        'issue': info['issue'],
        'allegiance': info['allegiance'],
        'affiliated': parse_affiliated(soup, slug),
    }

### Run the scrape
~3,500 characters, parallelized across 8 threads via `ThreadPoolExecutor`. `executor.map` preserves input order, so the output CSV stays alphabetical and checkpoints every 100 rows remain coherent. Tune `WORKERS` if the wiki starts throttling.

In [23]:
characters = pd.read_csv('../csvs/characters.csv').to_dict('records')
print(f'{len(characters)} characters to enrich')

COLUMNS = ['name', 'ID', 'father', 'mother', 'spouse', 'lover', 'issue', 'allegiance', 'affiliated']
OUT = '../csvs/characters_enriched.csv'
WORKERS = 8

rows = []
with ThreadPoolExecutor(max_workers=WORKERS) as executor:  # Speeds up runtime by 8x by parallelizing requests
    for i, row in enumerate(executor.map(scrape_character, characters), 1):
        rows.append(row)
        if i % 50 == 0:
            print(f'  {i}/{len(characters)}')
        if i % 100 == 0:
            pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)

pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)
print(f'Saved {len(rows)} rows to {OUT}')

3689 characters to enrich
  50/3689
  100/3689
  150/3689
  200/3689
  250/3689
  300/3689
  350/3689
  400/3689
  450/3689
  500/3689
  550/3689
  600/3689
  650/3689
  700/3689
  750/3689
  800/3689
  850/3689
  900/3689
  950/3689
  1000/3689
  1050/3689
  1100/3689
  1150/3689
  1200/3689
  1250/3689
  1300/3689
  1350/3689
  1400/3689
  1450/3689
  1500/3689
  1550/3689
  1600/3689
  1650/3689
  1700/3689
  1750/3689
  1800/3689
  1850/3689
  1900/3689
  1950/3689
  2000/3689
  2050/3689
  2100/3689
  2150/3689
  2200/3689
  2250/3689
  2300/3689
  2350/3689
  2400/3689
  2450/3689
  2500/3689
  2550/3689
  2600/3689
  2650/3689
  2700/3689
  2750/3689
  2800/3689
  2850/3689
  2900/3689
  2950/3689
  3000/3689
  3050/3689
  3100/3689
  3150/3689
  3200/3689
  3250/3689
  3300/3689
  3350/3689
  3400/3689
  3450/3689
  3500/3689
  3550/3689
  3600/3689
  3650/3689
Saved 3689 rows to characters_enriched.csv


## 3. Restrict slug-valued columns to characters only
The raw scraped columns hold every internal wiki link — characters, houses, battles, places, timeline pages, etc. For the network analysis we only want **character → character** edges, so we filter each slug-valued column against the set of all character IDs.

We key on `ID` (the canonical slug) rather than on display name, since many display names are ambiguous (several "Aegon Targaryen"s, several "Brandon Stark"s). The `ID` column already captures disambiguated pages like `Aegon_Targaryen_(son_of_Gaemon)` that a name-based lookup would collapse.

Free-text fallbacks (e.g. `Unknown`, `A butcher`, `Three sons [5]`) are dropped by the filter — they were only useful for display, and never participate in the graph.

In [24]:
df = pd.read_csv(OUT).fillna('')
character_ids = set(df['ID'])

SLUG_COLUMNS = ['father', 'mother', 'spouse', 'lover', 'issue', 'affiliated']


def keep_characters(cell):
    if not cell:
        return ''
    return ';'.join(s for s in cell.split(';') if s in character_ids)


def count_entries(col):
    return df[col].str.count(';').add(df[col].ne('').astype(int)).sum()


for col in SLUG_COLUMNS:
    before = count_entries(col)
    df[col] = df[col].map(keep_characters)
    after = count_entries(col)
    print(f'  {col:11s}: {before:>6} -> {after:<6}  (dropped {before - after})')

# `allegiance` is a separate namespace (houses / orders) — don't filter to character_ids,
# but drop free-text fallbacks so the column stays slug-only for joins against House data later.
before = count_entries('allegiance')
df['allegiance'] = df['allegiance'].map(
    lambda c: ';'.join(s for s in c.split(';') if s and ' ' not in s and '[' not in s) if c else ''
)
after = count_entries('allegiance')
print(f'  {"allegiance":11s}: {before:>6} -> {after:<6}  (dropped {before - after} free-text entries)')

df.to_csv(OUT, index=False)

  father     :    955 -> 875     (dropped 80)
  mother     :    515 -> 465     (dropped 50)
  spouse     :    547 -> 417     (dropped 130)
  lover      :    200 -> 184     (dropped 16)
  issue      :   1608 -> 1391    (dropped 217)
  affiliated : 145495 -> 56049   (dropped 89446)
  allegiance :   4111 -> 4103    (dropped 8 free-text entries)


### Quick sanity check

In [25]:
df = pd.read_csv(OUT)
print(df.shape)
df[df['name'].isin(['Eddard Stark', 'Jon Snow', 'Tyrion Lannister'])]
df[0:50]

(3689, 9)


,name,ID,father,mother,spouse,lover,issue,allegiance,affiliated
0,A certain man,A_certain_man,NaN,NaN,NaN,NaN,NaN,NaN,Varys;Tyrion_Lannister;Stannis_Baratheon;Cortn...
1,Abelar Hightower,Abelar_Hightower,NaN,NaN,NaN,NaN,NaN,House_Hightower,Valarr_Targaryen;Daeron_II_Targaryen;Duncan_th...
2,Abelon,Abelon,NaN,NaN,NaN,NaN,NaN,Citadel,NaN
3,Addam of Duskendale,Addam_of_Duskendale,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Addam Frey,Addam_Frey,NaN,NaN,NaN,NaN,NaN,House_Frey,Aerys_I_Targaryen;Lord_Frey;Ambrose_Butterwell...
5,Addam Hightower,Addam_Hightower,Manfred_Hightower_(Aegon's_Conquest),NaN,NaN,NaN,Manfred_Hightower_(son_of_Addam);Patrice_Hight...,House_Hightower;House_Tyrell,Manfred_Hightower_(Aegon's_Conquest);Manfred_H...
6,Addam Marbrand,Addam_Marbrand,Damon_Marbrand,NaN,NaN,NaN,NaN,House_Marbrand;House_Lannister;City_Watch_of_K...,Damon_Marbrand;Tywin_Lannister;Jaime_Lannister...
7,Addam Osgrey,Addam_Osgrey,Eustace_Osgrey,NaN,NaN,NaN,NaN,House_Osgrey,Eustace_Osgrey;Daeron_II_Targaryen;Wyman_Webbe...
8,Addam Rivers,Addam_Rivers,NaN,NaN,NaN,NaN,NaN,NaN,Arlan_III_Durrandon
9,Addam Velaryon,Addam_Velaryon,NaN,Marilda_of_Hull,NaN,NaN,NaN,House_Velaryon;House_Targaryen;Blacks,Laenor_Velaryon;Corlys_Velaryon;Marilda_of_Hul...
